# 03 — Building Permits: Preparation and Analysis

**Phase:** Building Permits Analysis  
**Notebook goal:** Understand the structure and category landscape of the permits dataset and establish a defensible residential-filtering rule before any aggregation is performed.  
**Status:** Exploration only — no filtering, no aggregations yet.

---

## 1. Purpose

This notebook begins the **Building Permits Analysis** block of the project.

The goal is to understand the category landscape of the Issued Building Permits dataset well enough to write a defensible residential-filtering rule. That rule will be applied in a later step before permit counts and project values are aggregated by year.

Specifically, this notebook:

1. Loads a 1,000-row sample of the raw permits file and confirms the required columns are present.
2. Reports the date range covered by the sample.
3. Explores the category columns — `TypeOfWork`, `PermitCategory`, `PropertyUse`, and `SpecificUseCategory` — to understand how permits are classified.
4. Identifies the decision that must be made before aggregation can proceed.

## 2. Analytical Grain

**One row = one permit record.**

Each row in the Issued Building Permits dataset represents a single building permit that was issued by the City of Vancouver. One property can have multiple permits issued at different times (for example, a demolition permit followed later by a new-construction permit).

**Permit issuance is a supply signal, not a completed housing unit.**

A permit is issued at the start of a regulated construction or renovation activity. It does not confirm that work was completed, that a unit was occupied, or that a dwelling was added to the housing stock. Permits are used here as a leading indicator of housing supply activity — they signal intent and approval, not delivery.

**The residential filtering rule must be based on observed categories, not assumptions.**

The dataset contains permits for residential, commercial, industrial, and institutional work. Before aggregating permit counts or project values as a housing supply signal, we must decide which rows to include. That decision must be grounded in the actual category values present in the data — not assumed in advance. This notebook exists to gather the evidence needed to make that decision.

## 3. Input

| Item | Detail |
|---|---|
| Source file | `data/raw/issued_building_permits_raw.csv` |
| Rows loaded | 1,000 (sample only) |
| Delimiter | `;` (semicolon) |
| Encoding | `utf-8-sig` (strips byte-order mark) |

Required columns for this analysis:

| Column | Role |
|---|---|
| `IssueDate` | Full date the permit was issued |
| `IssueYear` | Year extracted from `IssueDate` |
| `YearMonth` | Year-month string for finer time grouping |
| `ProjectValue` | Declared value of the permitted work |
| `TypeOfWork` | Broad work category (e.g. new construction, alteration) |
| `PermitCategory` | More specific permit classification |
| `PropertyUse` | High-level land use (e.g. Dwelling Uses) |
| `SpecificUseCategory` | Detailed use within the property use group |
| `GeoLocalArea` | City neighbourhood for geographic grouping |

### Import Libraries and Define Constants

We import `pandas` and `os`. Column names needed later in the notebook are stored as a list constant so they can be referenced consistently without retyping. The file path is built from the project root directory rather than hard-coded, so the notebook runs correctly regardless of where the repository is cloned.

In [ ]:
import os
import pandas as pd

BASE_DIR     = os.path.abspath(os.path.join(os.getcwd(), '..'))
PERMITS_PATH = os.path.join(BASE_DIR, 'data', 'raw', 'issued_building_permits_raw.csv')

REQUIRED_COLS = [
    'IssueDate',
    'IssueYear',
    'YearMonth',
    'ProjectValue',
    'TypeOfWork',
    'PermitCategory',
    'PropertyUse',
    'SpecificUseCategory',
    'GeoLocalArea',
]

SAMPLE_ROWS = 1_000

print('Setup complete.')
print(f'File path : {PERMITS_PATH}')

### Load a 1,000-Row Sample

We raise an explicit error if the file is missing before attempting to load, so the failure message is informative rather than a generic `FileNotFoundError` from pandas. After loading, we print the shape to confirm both the row count and column count are as expected.

In [ ]:
if not os.path.isfile(PERMITS_PATH):
    raise FileNotFoundError(
        f'Raw permits file not found. Expected: {PERMITS_PATH}'
    )

df = pd.read_csv(
    PERMITS_PATH,
    sep=';',
    nrows=SAMPLE_ROWS,
    encoding='utf-8-sig',
    low_memory=False,
)

print(f'File found  : {PERMITS_PATH}')
print(f'Sample shape: {df.shape}  →  ({df.shape[0]:,} rows, {df.shape[1]} columns)')

## 4. Initial Validation

Before exploring any distributions, we confirm that all required columns are present and report the missing-value rate for each. A required column that is absent or heavily null would change the analysis approach before any further work is done.

### Required Column Check

We loop over the list of required columns and check whether each one appears in the DataFrame. A `MISSING` status here means the column was not in the file at all — possibly renamed or not included in this export. A `FOUND` status means the column loaded correctly, though it may still contain null values (checked next).

In [ ]:
all_present = True
for col in REQUIRED_COLS:
    status = 'FOUND  ' if col in df.columns else 'MISSING'
    if status == 'MISSING':
        all_present = False
    print(f'  {status}: {col}')

print()
print(f'All required columns present: {all_present}')

### Missing-Value Counts for Required Columns

We count null values in each required column across the 1,000-row sample. Columns with high null rates may need special handling when the residential filter or aggregations are defined. `PermitCategory` in particular is worth watching — if it is heavily null, it may not be reliable as the sole basis for a residential filter.

In [ ]:
missing_counts = df[REQUIRED_COLS].isnull().sum()
missing_pct    = (missing_counts / len(df) * 100).round(1)

missing_summary = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct'  : missing_pct,
}).sort_values('missing_count', ascending=False)

display(missing_summary)

## 5. Date Coverage

We report the minimum and maximum `IssueYear` present in the sample. This tells us the approximate time span represented, which is important for deciding how many years of permit data will be available when the analysis is extended to the full dataset.

Because we are working with a 1,000-row sample, early or late years may be under-represented. The full date range should be verified when the full dataset is loaded.

In [ ]:
min_year = df['IssueYear'].min()
max_year = df['IssueYear'].max()
unique_years = sorted(df['IssueYear'].dropna().unique())

print(f'Minimum IssueYear in sample : {min_year}')
print(f'Maximum IssueYear in sample : {max_year}')
print(f'Unique years in sample      : {unique_years}')

## 6. Residential Classification Review

The following four columns are candidates for defining a residential filter. We inspect each one to understand the vocabulary used in the dataset. The output of this section is the evidence base for the decision in the next section.

For columns with a small number of unique values (`TypeOfWork`, `PermitCategory`), we display every value and its count. For columns with many unique values (`PropertyUse`, `SpecificUseCategory`), we display the top 20 by frequency. In all cases, null values are included in the count so we can see the full picture.

### TypeOfWork

`TypeOfWork` describes the broad category of construction activity being permitted. Common values across building permit datasets include new construction, additions, alterations, demolitions, and salvage or abatement work. Understanding which values appear here helps us decide whether this column alone is sufficient to identify residential supply, or whether it must be combined with a use-based column.

`value_counts(dropna=False)` is used throughout this section so null values appear in the table rather than being silently excluded.

In [ ]:
type_of_work_counts = df['TypeOfWork'].value_counts(dropna=False)
print(f'Unique values in TypeOfWork (including null): {df["TypeOfWork"].nunique(dropna=False)}')
print()
display(type_of_work_counts.to_frame('count'))

### PermitCategory

`PermitCategory` is a more specific classification than `TypeOfWork`. It may distinguish residential from commercial work within the same broad category, making it a strong candidate for a residential filter. However, if it has a high null rate (as suggested by the missing-value check above), any filter based solely on this column will produce a result with a significant gap where the category is unknown.

In [ ]:
permit_category_counts = df['PermitCategory'].value_counts(dropna=False)
print(f'Unique values in PermitCategory (including null): {df["PermitCategory"].nunique(dropna=False)}')
print()
display(permit_category_counts.to_frame('count'))

### PropertyUse

`PropertyUse` describes the intended use of the property to which the permit applies. A value of `Dwelling Uses` (or similar) is the clearest direct signal that a permit relates to residential housing. This column may be more consistently populated than `PermitCategory` and could serve as a reliable primary filter or as a complement to it.

We display the top 20 values by frequency, including nulls.

In [ ]:
property_use_counts = df['PropertyUse'].value_counts(dropna=False)
print(f'Unique values in PropertyUse (including null): {df["PropertyUse"].nunique(dropna=False)}')
print()
display(property_use_counts.head(20).to_frame('count'))

### SpecificUseCategory

`SpecificUseCategory` provides the most granular classification of the permitted use. Within `Dwelling Uses`, it may distinguish between single-detached houses, apartments, townhouses, and other dwelling types. This level of detail may be useful for disaggregating housing supply by type in a later analysis phase, or for excluding non-residential dwelling categories (such as care facilities or hotels).

We display the top 20 values by frequency, including nulls.

In [ ]:
specific_use_counts = df['SpecificUseCategory'].value_counts(dropna=False)
print(f'Unique values in SpecificUseCategory (including null): {df["SpecificUseCategory"].nunique(dropna=False)}')
print()
display(specific_use_counts.head(20).to_frame('count'))

## 7. Decision Required

Before aggregation can proceed, a residential-filtering rule must be defined and documented. The exploration above provides the evidence base. The decision has the following dimensions:

**Which column(s) define the residential filter?**

| Column | Strength | Risk |
|---|---|---|
| `PropertyUse` | Direct use signal; likely well-populated | May include non-housing dwelling uses |
| `PermitCategory` | Specific and descriptive | Significant null rate — gaps in coverage |
| `TypeOfWork` | Available but broad | Does not distinguish residential from commercial |
| `SpecificUseCategory` | Most granular | Many values; requires an explicit inclusion list |

**Which values within the chosen column(s) count as residential?**

The full-dataset value counts (not visible in this 1,000-row sample) are needed to confirm the complete vocabulary. The values observed here are indicative only.

**Should the filter target new construction only, or all permit types?**

For a housing supply signal, new-construction permits are the most direct measure. Addition and alteration permits add density to existing stock in a different way. The analysis scope should be clarified before the filter is set.

**Action required:** Document the chosen filter in `docs/dataset_decision_log.md` before implementing it in this notebook. The decision log entry should state the chosen column(s), the included values, the rationale, and any known limitations.

## 8. Next Step

Once the residential filter decision is documented in `docs/dataset_decision_log.md`, the following steps will be added to this notebook:

1. **Load the full dataset** — reload without `nrows` to work with all permit records.
2. **Apply the residential filter** — retain only rows matching the agreed category values.
3. **Validate the filter** — confirm the row count after filtering and review a sample of filtered-out rows to check for false positives and false negatives.
4. **Derive `permit_count_by_year`** — count residential permits issued per year.
5. **Derive `permit_project_value_by_year`** — sum `ProjectValue` for residential permits per year as an alternative supply-intensity measure.

Both aggregations will be saved to `data/processed/` for use in the cross-dataset analysis phase.